## Installation part

In [1]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import mediapy as media
import mujoco

In [2]:
np.set_printoptions(precision=3, suppress=True, linewidth=100)

#checking
try:
    mujoco.MjModel.from_xml_string('<mujoco/>')
    print("MuJoCo installed and verified successfully.")
except Exception as e:
    print(f"Installation check failed: {e}")

MuJoCo installed and verified successfully.


## MujoCo model setup..

In [3]:
xml = """
<mujoco>
  <worldbody>
    <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
    <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(xml)

In [ ]:
model.ngeom #number of geoms (here its spahere and box)

2

In [5]:
model.geom_rgba

array([[1., 0., 0., 1.],
       [0., 1., 0., 1.]], dtype=float32)

In [7]:
# model.geom('green_sphere')
model.geom('green_sphere').rgba

array([0., 1., 0., 1.], dtype=float32)

In [ ]:
#longer version..
id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, 'green_sphere')
model.geom_rgba[id, :]

array([0., 1., 0., 1.], dtype=float32)

In [13]:
print('id of "green_sphere": ', model.geom('green_sphere').id)
print('id of "red_box": ', model.geom('red_box').id)

print('name of geom 1: ', model.geom(1).name)
print('name of body 0: ', model.geom(0).name)

id of "green_sphere":  1
id of "red_box":  0
name of geom 1:  green_sphere
name of body 0:  red_box


In [ ]:
data = mujoco.MjData(model) #containts state information (like positions of 2 geoms, shown below)

In [17]:
print(data.geom_xpos)

[[0. 0. 0.]
 [0. 0. 0.]]


In [19]:
#but it shows at origin, despite assigning position to this in the model xml definition
mujoco.mj_kinematics(model, data) # a function (in this case -> mj_kinematics, needs to be used)
print('raw access:\n', data.geom_xpos)

# MjData also supports named access:
print('\nnamed access:\n', data.geom('green_sphere').xpos)

#now it should print correctly

raw access:
 [[0.  0.  0. ]
 [0.2 0.2 0.2]]

named access:
 [0.2 0.2 0.2]


## Rendering and actually displaying

In [5]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "9"
os.environ["MUJOCO_GL"] = "egl"

import time
import numpy as np
import matplotlib.pyplot as plt
import mediapy as media
import mujoco

In [2]:
!pwd

/data/pbk5339/thesis_new/demo


In [4]:
#rewriting the model def

xml = """
<mujoco>
  <worldbody>
    <light name="top" pos="0 0 1"/>
    <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
    <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
  </worldbody>
</mujoco>
"""
# Make model and data
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

# mujoco.mj_step(model, data) -> #not needed if at rest!!

# Make renderer, render and show the pixels
with mujoco.Renderer(model) as renderer:
  mujoco.mj_forward(model, data)
  renderer.update_scene(data)
  pixels = renderer.render()
  
  media.write_image("output.png", pixels)

print("Saved frame to output.png")

Saved frame to output.png


In [6]:
#to run video..

duration = 3.8 
framerate = 60

frames = []
mujoco.mj_resetData(model, data) #resets teh state and time

with mujoco.Renderer(model) as renderer:
    while data.time < duration:
        mujoco.mj_step(model, data) #steps to next state? (2 milliseconds..)
        if len(frames) < data.time * framerate: #calling "update_scene" (graphics renderer) only after 3.8 secs (not 2ms)
            renderer.update_scene(data)
            pixels = renderer.render()
            frames.append(pixels)
            
output_path = "simulation.mp4"
media.write_video(output_path, frames, fps=framerate)

print(f"Video saved successfully to {output_path}")

Video saved successfully to simulation.mp4


### creating a moving video..(add joints, DOFs to model XML)

In [7]:
xml = """
<mujoco>
  <worldbody>
    <light name="top" pos="0 0 1"/>
    <body name="box_and_sphere" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
      <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
      <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)


#for visualizing joints
scene_option = mujoco.MjvOption()
scene_option.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

duration = 3.8  # (seconds)
framerate = 60  # (Hz)

frames = []
mujoco.mj_resetData(model, data) #resets teh state and time

with mujoco.Renderer(model) as renderer:
    while data.time < duration:
        mujoco.mj_step(model, data) #steps to next state? (2 milliseconds..)
        if len(frames) < data.time * framerate: #calling "update_scene" (graphics renderer) only after 3.8 secs (not 2ms)
            renderer.update_scene(data)
            pixels = renderer.render()
            frames.append(pixels)
            
output_path = "simulation.mp4"
media.write_video(output_path, frames, fps=framerate)

print(f"Video saved successfully to {output_path}")

Video saved successfully to simulation.mp4


In [ ]:
print('default gravity', model.opt.gravity) #accessing an "option"


default gravity [ 0.    0.   -9.81]


### tippe top model (top toy)

In [11]:
#new model-

tippe_top = """
<mujoco model="tippe top">
  <option integrator="RK4"/>

  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3"
     rgb2=".2 .3 .4" width="300" height="300"/>
    <material name="grid" texture="grid" texrepeat="8 8" reflectance=".2"/>
  </asset>

  <worldbody>
    <geom size=".2 .2 .01" type="plane" material="grid"/>
    <light pos="0 0 .6"/>
    <camera name="closeup" pos="0 -.1 .07" xyaxes="1 0 0 0 1 2"/>
    <body name="top" pos="0 0 .02">
      <freejoint/>
      <geom name="ball" type="sphere" size=".02" />
      <geom name="stem" type="cylinder" pos="0 0 .02" size="0.004 .008"/>
      <geom name="ballast" type="box" size=".023 .023 0.005"  pos="0 0 -.015"
       contype="0" conaffinity="0" group="3"/>
    </body>
  </worldbody>

  <keyframe>
    <key name="spinning" qpos="0 0 0.02 1 0 0 0" qvel="0 0 0 0 1 200" />
  </keyframe>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(tippe_top)
data = mujoco.MjData(model)

with mujoco.Renderer(model) as renderer:
  mujoco.mj_forward(model, data)
  renderer.update_scene(data, camera="closeup")
  pixels = renderer.render()
  
  media.write_image("tippe_top.png", pixels)

print("Saved frame to tippe_top.png")

Saved frame to tippe_top.png


In [ ]:
print('positions', data.qpos)
print('velocities', data.qvel)


'''
#NOTE:

data.qpos -> [0.0,  0.0,  0.02,   1.0,  0.0,  0.0,  0.0]
             |_____ x, y, z ____|  |___ quaternion (w, x, y, z) ___|
'''

positions [0.   0.   0.02 1.   0.   0.   0.  ]
velocities [0. 0. 0. 0. 0. 0.]


In [12]:
duration = 7    # (seconds)
framerate = 60  # (Hz)

# Simulate and display video.
frames = []
mujoco.mj_resetDataKeyframe(model, data, 0)  # Reset the state to keyframe 0
with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * framerate:
      renderer.update_scene(data, "closeup")
      pixels = renderer.render()
      frames.append(pixels)

output_path = "tippe_top.mp4"
media.write_video(output_path, frames, fps=framerate)

print(f"Video saved successfully to {output_path}")

Video saved successfully to tippe_top.mp4
